# Loss minimization

_Artificial Intelligence (CS221)_

**Measure how wrong the model is. Then pick weights that make that number smallest.**

A loss is a number that says how wrong one prediction is. Small loss is good. Big loss is bad.

     The training loss is the average loss over all examples.

---

This notebook is a practice scaffold for the **Loss minimization** lesson. We'll compare three loss functions on the same handful of margins, then watch how each one judges a real classifier's predictions on tumour scans. Run each cell, read the note above it, then try the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

Every loss here is a function of the **margin** `m = score · true_label`: positive means the prediction is correct, and larger means more confident. We compare three classic losses on the same five margins, in two steps: (1) lay out the margins, (2) compute each loss and its average.

### Step 1 — Lay out a few margins

Rather than re-run a model, we hand-pick five margins that cover the interesting cases: two comfortably correct (`2.0`, `1.5`), one barely correct (`0.3`), and two wrong (`-0.5`, `-1.2`). Watching how each loss treats *these specific points* is the whole lesson.

In [ ]:
# Margin m = score * true_label. Positive = correct; bigger = more confident.
m = np.array([2.0, 0.3, -0.5, 1.5, -1.2])

print("margins:", m)

### Step 2 — Compute three losses and their averages

- **Zero-one loss** counts a mistake (`1`) whenever the margin is negative — it measures pure accuracy but is flat and non-differentiable, so it can't guide gradient steps.
- **Hinge loss** is `max(0, 1 − m)`: zero once the margin clears `1`, otherwise growing linearly. It is the SVM loss and rewards a *safety buffer*, not just being right.
- **Squared loss** `(m − 1)²` punishes any distance from a margin of `1` in *both* directions — so it even penalises over-confident-correct points, which is usually undesirable here.

The **mean** of each is the training loss we would actually minimise.

In [ ]:
zero_one = (m < 0).astype(float)        # 1 if wrong, else 0
hinge = np.maximum(1 - m, 0)            # SVM loss: 0 once the margin passes 1
squared = (m - 1) ** 2                  # punishes any distance from a margin of 1

print("zero-one:", zero_one, " mean =", zero_one.mean())
print("hinge   :", np.round(hinge, 2), " mean =", round(hinge.mean(), 3))
print("squared :", np.round(squared, 2), " mean =", round(squared.mean(), 3))

## Visualize the data & results

_On real breast-cancer scans, how harshly does each loss punish the model's margins?_

Now the margins come from an actual fitted classifier instead of being made up. We compute the margin for every scan, then summarise the three losses across all 569 of them.

### First, look at the data

Before we train on the breast-cancer scans, here's the full dataset — every feature column, the two class labels, and a few real rows. (The training code below uses only a couple of these columns so the result is easy to plot.)

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

### Step 1 — Fit a classifier and compute every scan's margin

We fit `LogisticRegression` on two standardised features, then form the margin for each scan as `(w · x + b) · (2y − 1)`. The `2y − 1` term remaps the `{0, 1}` labels to `{−1, +1}` so the sign convention matches the loss definitions above.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X = data.data[:, [0, 1]]            # mean radius, mean texture
y = data.target                    # 1 = benign, 0 = malignant

scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)
clf = LogisticRegression().fit(X_scaled, y)

w, b = clf.coef_[0], clf.intercept_[0]
signed_label = 2 * y - 1           # remap {0,1} labels to {-1,+1}
margin = (X_scaled @ w + b) * signed_label

print("computed margins for", len(margin), "scans")

### Step 2 — Pick five named scans and summarise all three losses

We single out five scans to show the **hinge loss per scan** (zero when the margin already exceeds `1`, positive otherwise). Then, across *all* scans, we average each loss to compare how severe each one is overall. Note hinge and squared stay positive even when accuracy is high — they care about margin, not just the sign.

In [ ]:
selected = [0, 5, 20, 50, 100]
margins_sel = margin[selected]
hinge_sel = np.maximum(1 - margins_sel, 0)

mean_zero_one = (margin < 0).mean()
mean_hinge = np.maximum(1 - margin, 0).mean()
mean_squared = ((margin - 1) ** 2).mean()

print("mean zero-one:", round(mean_zero_one, 3))
print("mean hinge   :", round(mean_hinge, 3))
print("mean squared :", round(mean_squared, 3))

### Step 3 — Plot per-scan hinge loss and the mean losses

Left: the hinge loss for the five named scans — green bars are scans already past the margin of `1` (zero loss), red bars are scans the model should push harder on. Right: the three mean losses side by side, so you can see how differently each loss scores the very same model.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left — hinge loss on the five named scans
bar_colors = ["#7ee787" if h == 0 else "#ff7b72" for h in hinge_sel]
ax1.bar(range(5), hinge_sel, color=bar_colors)
ax1.set_xticks(range(5))
ax1.set_xticklabels([str(i + 1) for i in selected])
ax1.set_title("hinge loss on 5 named scans")
ax1.set_xlabel("scan id")

# Right — mean of each loss over all 569 scans
ax2.bar(["zero-one", "hinge", "squared"],
        [mean_zero_one, mean_hinge, mean_squared],
        color=["#4ea1ff", "#ffb454", "#c89bff"])
ax2.set_title("mean loss over 569 scans")

plt.tight_layout()
plt.show()